In [1]:
#!/usr/bin/env python3
"""
download_manshurat_pdfs.py
يحمل ملفات PDF من صفحة الأحكام بموقع manshurat.org/taxonomy/term/16
اعمل: pip install requests beautifulsoup4 tqdm
ثم شغل: python download_manshurat_pdfs.py
"""

import os
import time
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

BASE_URL = "https://manshurat.org"
START_URL = "https://manshurat.org/taxonomy/term/16"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; DownloaderBot/1.0; +https://example.com/bot)"
}
DOWNLOAD_DIR = "downloads"
DELAY_BETWEEN_REQUESTS = 1.0  # ثانية; لو عايزة تخففي أو تزودي
MAX_DOCS = None  # حطي رقم للتحديد، أو None لتحميل الكل

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

session = requests.Session()
session.headers.update(HEADERS)


def get_soup(url):
    r = session.get(url, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def find_pager_links(soup, current_url):
    """يبحث روابط صفحات الترقيم (pagination) ويرجع قائمة روابط فريدة"""
    links = set()
    pager = soup.select("ul.pager a, .pager a, nav.pager a")
    for a in pager:
        href = a.get("href")
        if href:
            links.add(urljoin(current_url, href))
    return sorted(links)


def extract_document_page_links(listing_soup, base_url):
    """من صفحة قائمة الأحكام يستخرج روابط صفحات كل حكم/وثيقة"""
    links = set()
    # يعتمد على بنية الموقع: عادة المقالات داخل عناصر article أو h2 a أو .views-row a
    selectors = [
        "article a",         # generic
        ".views-row a",       # drupal views
        ".node-title a",
        "h2 a",
        ".views-field-title a"
    ]
    for sel in selectors:
        for a in listing_soup.select(sel):
            href = a.get("href")
            if href and href.startswith("/"):
                links.add(urljoin(base_url, href))
            elif href and href.startswith("http"):
                links.add(href)
    return sorted(links)


def find_pdf_links_in_doc(doc_soup, doc_url):
    """في صفحة الوثيقة يجيب جميع روابط ال PDF"""
    pdfs = set()
    # أول: أبحث عن <a> الذي يحتوي .pdf
    for a in doc_soup.find_all("a", href=True):
        href = a["href"]
        if ".pdf" in href.lower():
            pdfs.add(urljoin(doc_url, href))
    # ثانياً: أبحث داخل iframe أو embed
    for tag in doc_soup.find_all(["iframe", "embed"]):
        src = tag.get("src") or tag.get("data")
        if src and ".pdf" in src.lower():
            pdfs.add(urljoin(doc_url, src))
    return sorted(pdfs)


def sanitize_filename(url):
    """يحاول استخراج اسم ملف معقول من الرابط"""
    path = urlparse(url).path
    name = os.path.basename(path)
    if not name:
        name = "document.pdf"
    # إزالة محارف غير مرغوبة
    name = name.split("?")[0]
    return name


def download_file(url, target_folder):
    local_name = sanitize_filename(url)
    local_path = os.path.join(target_folder, local_name)
    # إذا الملف موجود نتخطاه
    if os.path.exists(local_path):
        return local_path, False
    with session.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = r.headers.get('content-length')
        if total is None:
            # بدون طول معروف
            with open(local_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        else:
            total = int(total)
            with open(local_path, "wb") as f, tqdm(
                total=total, unit="B", unit_scale=True, desc=local_name
            ) as pbar:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
    return local_path, True


def main():
    to_visit = [START_URL]
    visited_listing = set()
    doc_pages_seen = set()
    pdf_links_seen = set()
    downloaded = 0

    # اجمع روابط كل صفحات الترقيم تلقائياً
    i = 0
    while i < len(to_visit):
        url = to_visit[i]
        i += 1
        if url in visited_listing:
            continue
        print(f"جاري معالجة صفحة قائمة: {url}")
        try:
            soup = get_soup(url)
        except Exception as e:
            print("خطأ في جلب الصفحة:", e)
            continue
        visited_listing.add(url)

        # استخراج صفحات الوثائق من صفحة قائمة النتائج
        docs = extract_document_page_links(soup, BASE_URL)
        for d in docs:
            if d not in doc_pages_seen:
                doc_pages_seen.add(d)

        # إضافة صفحات الترقيم (pager) ليتم زيارتها لاحقًا
        pager_links = find_pager_links(soup, url)
        for p in pager_links:
            if p not in visited_listing and p not in to_visit:
                to_visit.append(p)

        time.sleep(DELAY_BETWEEN_REQUESTS)

    print(f"عدد صفحات الوثائق المكتشفة: {len(doc_pages_seen)}")

    # تصفح كل صفحة وثيقة للعثور على PDF وتحميلها
    for doc_url in list(doc_pages_seen):
        if MAX_DOCS and downloaded >= MAX_DOCS:
            break
        try:
            dsoup = get_soup(doc_url)
        except Exception as e:
            print("خطأ في فتح صفحة الوثيقة", doc_url, e)
            continue

        pdfs = find_pdf_links_in_doc(dsoup, doc_url)
        if not pdfs:
            # بعض الصفحات تعرض ملفًا داخل زر تحميل أو ضمن JSON — ننظر لنصوص تحتوي 'pdf' أيضًا
            text = dsoup.get_text()
            if ".pdf" in text.lower():
                # حاول أن تجد URL كامل داخل النص
                import re
                matches = re.findall(r"https?://[^\s'\"<>]+\.pdf", text, flags=re.IGNORECASE)
                for m in matches:
                    pdfs.append(m)

        for pdf in pdfs:
            if pdf in pdf_links_seen:
                continue
            pdf_links_seen.add(pdf)
            print("تحميل:", pdf)
            try:
                path, new = download_file(pdf, DOWNLOAD_DIR)
                if new:
                    downloaded += 1
                print("تم:", path)
            except Exception as e:
                print("فشل تحميل:", pdf, e)
            time.sleep(DELAY_BETWEEN_REQUESTS)

    print("انتهى. عدد الملفات التي تم تنزيلها:", downloaded)


if __name__ == "__main__":
    main()


جاري معالجة صفحة قائمة: https://manshurat.org/taxonomy/term/16
عدد صفحات الوثائق المكتشفة: 75
تحميل: https://docs.google.com/gview?embedded=true&url=https%3A%2F%2Fmanshurat.org%2Fsites%2Fdefault%2Ffiles%2F1-brqm_32_lsn_44_qdyy.pdf
تم: downloads/gview
خطأ في فتح صفحة الوثيقة http://twitter.com/intent/tweet?url=https%3A%2F%2Fmanshurat.org%2Fcontent%2Fdm-dstwry-fqr-bqnwn-lmrft-lmdny-wltjry-mnht-qd-lmhkm-musdr-lhkm-hq-nzr-lltms&text=%D8%B9%D8%AF%D9%85%20%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%D9%8A%D8%A9%20%D9%81%D9%82%D8%B1%D8%A9%20%D8%A8%D9%82%D8%A7%D9%86%D9%88%D9%86%20%D8%A7%D9%84%D9%85%D8%B1%D8%A7%D9%81%D8%B9%D8%A7%D8%AA%20%D8%A7%D9%84%D9%85%D8%AF%D9%86%D9%8A%D8%A9%20%D9%88%D8%A7%D9%84%D8%AA%D8%AC%D8%A7%D8%B1%D9%8A%D8%A9%20%D9%85%D9%86%D8%AD%D8%AA%20%D9%82%D8%B6%D8%A7%D8%A9%20%D8%A7%D9%84%D9%85%D8%AD%D9%83%D9%85%D8%A9%20%D9%85%D9%8F%D8%B5%D8%AF%D8%B1%D8%A9%20%D8%A7%D9%84%D8%AD%D9%83%D9%85%20%D8%AD%D9%82%20%D9%86%D8%B8%D8%B1%20%D8%A7%D9%84%D8%A7%D9%84%D8%AA%D9%85%D8%A7%D8%B3%20 520 Server Error:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import unquote, urlparse, parse_qs

BASE_URL = "https://manshurat.org"
START_URL = "https://manshurat.org/taxonomy/term/16"
DOWNLOAD_DIR = "downloads/pdf"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

def extract_real_pdf(url):
    """لو اللينك gview نرجّعه للـ pdf الأصلي"""
    if "gview" in url:
        parsed = urlparse(url)
        real_url = parse_qs(parsed.query).get("url", [""])[0]
        return unquote(real_url)
    return url

def download_pdf(url):
    response = requests.get(url, stream=True)
    if response.status_code == 200 and url.endswith(".pdf"):
        filename = url.split("/")[-1]
        filepath = os.path.join(DOWNLOAD_DIR, filename)
        with open(filepath, "wb") as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
        print(f"✔ تم التحميل: {filename}")
    else:
        print(f"✖ فشل التحميل: {url}")

def process_page(url):
    print(f"\n📄 معالجة الصفحة: {url}")
    html = requests.get(url).text
    soup = BeautifulSoup(html, "html.parser")

    # استخراج روابط الوثائق
    for a in soup.find_all("a", href=True):
        href = a["href"]

        # تجاهل تويتر + لينكات مش وثائق
        if "twitter.com" in href or "intent" in href:
            continue

        # لو فيه gview
        if "gview" in href:
            pdf_url = extract_real_pdf(href)
            if pdf_url.endswith(".pdf"):
                download_pdf(pdf_url)
            continue

        # روابط pdf مباشرة
        if href.endswith(".pdf"):
            if href.startswith("/"):
                href = BASE_URL + href
            download_pdf(href)

    # الانتقال للصفحة التالية إن وجدت
    next_link = soup.find("a", rel="next")
    if next_link:
        next_page = BASE_URL + next_link["href"]
        process_page(next_page)

# ابدأ
process_page(START_URL)



📄 معالجة الصفحة: https://manshurat.org/taxonomy/term/16


In [4]:
!pip install aiohttp aiofiles tqdm beautifulsoup4

In [6]:
# save_pdfs_async.py
import asyncio
import aiohttp
import aiofiles
import os
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
nest_asyncio.apply()

await main()

ROOT_PAGE = "https://manshurat.org/taxonomy/term/16"
OUTPUT_DIR = "downloaded_pdfs"
CONCURRENT = 8
RATE_MS = 200  # تأخير بسيط بين الطلبات لكل عملية (ملّي ثانية)

def make_filename_from_url(url):
    parsed = urlparse(url)
    name = os.path.basename(parsed.path)
    if not name:
        name = "file.pdf"
    if not name.lower().endswith(".pdf"):
        name += ".pdf"
    return "".join(c for c in name if c.isalnum() or c in "._-")

async def fetch_html(session, url):
    async with session.get(url, timeout=30) as resp:
        resp.raise_for_status()
        return await resp.text()

async def download_pdf(session, url, out_dir, sem):
    filename = make_filename_from_url(url)
    path = os.path.join(out_dir, filename)
    if os.path.exists(path):
        return f"skipped: {filename}"
    async with sem:
        try:
            async with session.get(url, timeout=60) as r:
                r.raise_for_status()
                f = await aiofiles.open(path, mode='wb')
                async for chunk in r.content.iter_chunked(1024*8):
                    await f.write(chunk)
                await f.close()
            await asyncio.sleep(RATE_MS / 1000.0)
            return f"downloaded: {filename}"
        except Exception as e:
            return f"failed: {url} -> {e}"

async def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    headers = {"User-Agent": "pdf-downloader/1.0"}
    sem = asyncio.Semaphore(CONCURRENT)
    async with aiohttp.ClientSession(headers=headers) as session:
        html = await fetch_html(session, ROOT_PAGE)
        soup = BeautifulSoup(html, "html.parser")
        links = []
        for a in soup.find_all("a", href=True):
            href = a["href"].strip()
            full = urljoin(ROOT_PAGE, href)
            if full.lower().endswith(".pdf") or "application/pdf" in a.get("type","").lower():
                links.append(full)
        print(f"Found {len(links)} pdf links.")
        tasks = [download_pdf(session, url, OUTPUT_DIR, sem) for url in links]
        for coro in asyncio.as_completed(tasks):
            res = await coro
            print(res)

if __name__ == "__main__":
    asyncio.run(main())


Found 0 pdf links.
Found 0 pdf links.


In [7]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

BASE = "https://manshurat.org"
START_PAGE = "https://manshurat.org/taxonomy/term/16"
OUTPUT_DIR = "manshurat_pdfs"
HEADERS = {"User-Agent": "pdf-downloader/1.0"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_pdf_links(page_url):
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".pdf"):
            full = urljoin(BASE, href)
            links.append(full)
    return links

def get_pagination_links(page_url):
    # حسب تصميم منشورات، ابحث على روابط الصفحات التالية
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")
    pages = []
    for a in soup.select("ul.pagination li a[href]"):
        pages.append(urljoin(BASE, a["href"]))
    return pages

visited = set()
to_visit = [START_PAGE]
all_pdfs = set()

while to_visit:
    url = to_visit.pop(0)
    if url in visited: continue
    visited.add(url)
    print("Visiting", url)
    try:
        pdfs = get_pdf_links(url)
        for p in pdfs:
            all_pdfs.add(p)
        # إجلب صفحات متتابعة (pagination) إن وجدت
        next_pages = get_pagination_links(url)
        for np in next_pages:
            if np not in visited:
                to_visit.append(np)
    except Exception as e:
        print("Failed to parse", url, e)
    time.sleep(1)  # تأخير بسيط لتقليل الضغط

print(f"Found {len(all_pdfs)} pdf links.")

for pdf_url in all_pdfs:
    try:
        r = requests.get(pdf_url, headers=HEADERS, stream=True, timeout=30)
        r.raise_for_status()
        name = os.path.basename(pdf_url)
        path = os.path.join(OUTPUT_DIR, name)
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Saved", name)
    except Exception as e:
        print("Failed to download", pdf_url, e)


Visiting https://manshurat.org/taxonomy/term/16


/usr/local/lib/python3.12/dist-packages/bs4/builder/__init__.py:411: RuntimeWarning: coroutine 'main' was never awaited
  for attr in list(modified_attrs.keys()):


Visiting https://manshurat.org/taxonomy/term/16?page=1
Visiting https://manshurat.org/taxonomy/term/16?page=2
Visiting https://manshurat.org/taxonomy/term/16?page=3
Visiting https://manshurat.org/taxonomy/term/16?page=4
Visiting https://manshurat.org/taxonomy/term/16?page=5
Visiting https://manshurat.org/taxonomy/term/16?page=6
Visiting https://manshurat.org/taxonomy/term/16?page=7
Visiting https://manshurat.org/taxonomy/term/16?page=8
Visiting https://manshurat.org/taxonomy/term/16?page=156
Visiting https://manshurat.org/taxonomy/term/16?page=9
Visiting https://manshurat.org/taxonomy/term/16?page=10
Visiting https://manshurat.org/taxonomy/term/16?page=11
Visiting https://manshurat.org/taxonomy/term/16?page=12
Visiting https://manshurat.org/taxonomy/term/16?page=155
Visiting https://manshurat.org/taxonomy/term/16?page=148
Visiting https://manshurat.org/taxonomy/term/16?page=149
Visiting https://manshurat.org/taxonomy/term/16?page=150
Visiting https://manshurat.org/taxonomy/term/16?page

In [8]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

BASE = "https://manshurat.org"
START_PAGE = "https://manshurat.org/taxonomy/term/16?page=1"
OUTPUT_DIR = "manshurat_pdfs"
HEADERS = {"User-Agent": "pdf-downloader/1.0"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_pdf_links(page_url):
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".pdf"):
            full = urljoin(BASE, href)
            links.append(full)
    return links

def get_pagination_links(page_url):
    # حسب تصميم منشورات، ابحث على روابط الصفحات التالية
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")
    pages = []
    for a in soup.select("ul.pagination li a[href]"):
        pages.append(urljoin(BASE, a["href"]))
    return pages

visited = set()
to_visit = [START_PAGE]
all_pdfs = set()

while to_visit:
    url = to_visit.pop(0)
    if url in visited: continue
    visited.add(url)
    print("Visiting", url)
    try:
        pdfs = get_pdf_links(url)
        for p in pdfs:
            all_pdfs.add(p)
        # إجلب صفحات متتابعة (pagination) إن وجدت
        next_pages = get_pagination_links(url)
        for np in next_pages:
            if np not in visited:
                to_visit.append(np)
    except Exception as e:
        print("Failed to parse", url, e)
    time.sleep(1)  # تأخير بسيط لتقليل الضغط

print(f"Found {len(all_pdfs)} pdf links.")

for pdf_url in all_pdfs:
    try:
        r = requests.get(pdf_url, headers=HEADERS, stream=True, timeout=30)
        r.raise_for_status()
        name = os.path.basename(pdf_url)
        path = os.path.join(OUTPUT_DIR, name)
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Saved", name)
    except Exception as e:
        print("Failed to download", pdf_url, e)


Visiting https://manshurat.org/taxonomy/term/16?page=1
Visiting https://manshurat.org/taxonomy/term/16
Visiting https://manshurat.org/taxonomy/term/16?page=2
Visiting https://manshurat.org/taxonomy/term/16?page=3
Visiting https://manshurat.org/taxonomy/term/16?page=4
Visiting https://manshurat.org/taxonomy/term/16?page=5
Visiting https://manshurat.org/taxonomy/term/16?page=6
Visiting https://manshurat.org/taxonomy/term/16?page=7
Visiting https://manshurat.org/taxonomy/term/16?page=8


KeyboardInterrupt: 

In [9]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

BASE = "https://manshurat.org"
START_PAGE = "https://manshurat.org/taxonomy/term/16"
OUTPUT_DIR = "manshurat_pdfs"
HEADERS = {"User-Agent": "pdf-downloader/1.0"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_article_links(page_url):
    """اجمع روابط المقالات داخل صفحة القسم"""
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    # روابط المقالات غالبًا في h2/a أو div.entry-title a
    for a in soup.select("h2 a, div.views-field-title a, article a"):
        href = a.get("href")
        if href:
            full = urljoin(BASE, href)
            links.append(full)
    return links

def get_pdf_links(article_url):
    """اجمع روابط PDF داخل صفحة المقالة"""
    resp = requests.get(article_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".pdf"):
            full = urljoin(BASE, href)
            links.append(full)
    return links

# ====== جمع جميع روابط المقالات من الصفحات ======
visited_pages = set()
to_visit_pages = [START_PAGE]
all_articles = set()

while to_visit_pages:
    url = to_visit_pages.pop(0)
    if url in visited_pages:
        continue
    visited_pages.add(url)
    print("Visiting section page:", url)
    try:
        articles = get_article_links(url)
        for a in articles:
            all_articles.add(a)
        # تحقق من وجود pagination
        resp = requests.get(url, headers=HEADERS, timeout=30)
        soup = BeautifulSoup(resp.text, "html.parser")
        for a in soup.select("ul.pagination li a[href]"):
            next_page = urljoin(BASE, a["href"])
            if next_page not in visited_pages:
                to_visit_pages.append(next_page)
    except Exception as e:
        print("Failed page:", url, e)
    time.sleep(1)

print(f"Found {len(all_articles)} articles. Now fetching PDFs...")

# ====== زيارة كل مقال لجمع PDF ======
all_pdfs = set()
for article in all_articles:
    try:
        pdfs = get_pdf_links(article)
        all_pdfs.update(pdfs)
    except Exception as e:
        print("Failed article:", article, e)
    time.sleep(0.5)

print(f"Found {len(all_pdfs)} PDF files. Downloading...")

# ====== تحميل الملفات ======
for pdf_url in all_pdfs:
    try:
        r = requests.get(pdf_url, headers=HEADERS, stream=True, timeout=60)
        r.raise_for_status()
        name = os.path.basename(pdf_url)
        path = os.path.join(OUTPUT_DIR, name)
        if os.path.exists(path):
            print("Skipped:", name)
            continue
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Saved:", name)
    except Exception as e:
        print("Failed download:", pdf_url, e)


Visiting section page: https://manshurat.org/taxonomy/term/16


/usr/local/lib/python3.12/dist-packages/bs4/element.py:377: RuntimeWarning: coroutine 'main' was never awaited
  def setup(


Visiting section page: https://manshurat.org/taxonomy/term/16?page=1
Visiting section page: https://manshurat.org/taxonomy/term/16?page=2
Visiting section page: https://manshurat.org/taxonomy/term/16?page=3
Visiting section page: https://manshurat.org/taxonomy/term/16?page=4
Visiting section page: https://manshurat.org/taxonomy/term/16?page=5
Visiting section page: https://manshurat.org/taxonomy/term/16?page=6
Visiting section page: https://manshurat.org/taxonomy/term/16?page=7
Visiting section page: https://manshurat.org/taxonomy/term/16?page=8
Visiting section page: https://manshurat.org/taxonomy/term/16?page=156
Visiting section page: https://manshurat.org/taxonomy/term/16?page=9
Visiting section page: https://manshurat.org/taxonomy/term/16?page=10
Visiting section page: https://manshurat.org/taxonomy/term/16?page=11
Visiting section page: https://manshurat.org/taxonomy/term/16?page=12
Visiting section page: https://manshurat.org/taxonomy/term/16?page=155
Visiting section page: http

KeyboardInterrupt: 

In [10]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import warnings

# ====== تجاهل تحذيرات RuntimeWarning في Jupyter ======
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ====== إعدادات ======
BASE = "https://manshurat.org"
START_PAGE = "https://manshurat.org/taxonomy/term/16"
OUTPUT_DIR = "manshurat_pdfs"
HEADERS = {"User-Agent": "pdf-downloader/1.0"}
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ====== دوال مساعدة ======
def get_article_links(page_url):
    """اجمع روابط المقالات داخل صفحة القسم"""
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    # كل المقالات غالبًا داخل div.views-row أو article
    for container in soup.select("div.views-row, article"):
        a_tag = container.find("a", href=True)
        if a_tag:
            full = urljoin(BASE, a_tag['href'])
            links.append(full)
    return links

def get_pdf_links(article_url):
    """اجمع روابط PDF داخل صفحة المقالة"""
    resp = requests.get(article_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".pdf"):
            full = urljoin(BASE, href)
            links.append(full)
    return links

def get_pagination_links(page_url):
    """اجمع روابط الصفحات التالية (pagination)"""
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")
    pages = []
    for a in soup.select("ul.pagination li a[href]"):
        full = urljoin(BASE, a["href"])
        pages.append(full)
    return pages

# ====== زحف صفحات القسم ======
visited_pages = set()
to_visit_pages = [START_PAGE]
all_articles = set()

while to_visit_pages:
    url = to_visit_pages.pop(0)
    if url in visited_pages:
        continue
    visited_pages.add(url)
    print("Visiting section page:", url)
    try:
        # جمع المقالات
        articles = get_article_links(url)
        for a in articles:
            all_articles.add(a)
        # جمع صفحات pagination
        next_pages = get_pagination_links(url)
        for np in next_pages:
            if np not in visited_pages and np not in to_visit_pages:
                to_visit_pages.append(np)
    except Exception as e:
        print("Failed page:", url, e)
    time.sleep(0.5)  # تأخير بسيط لتقليل الضغط

print(f"Found {len(all_articles)} articles. Now fetching PDFs...")

# ====== زيارة كل مقال لجمع PDF ======
all_pdfs = set()
for article in all_articles:
    try:
        pdfs = get_pdf_links(article)
        all_pdfs.update(pdfs)
    except Exception as e:
        print("Failed article:", article, e)
    time.sleep(0.2)

print(f"Found {len(all_pdfs)} PDF files. Downloading...")

# ====== تحميل الملفات ======
for pdf_url in all_pdfs:
    try:
        r = requests.get(pdf_url, headers=HEADERS, stream=True, timeout=60)
        r.raise_for_status()
        name = os.path.basename(pdf_url)
        path = os.path.join(OUTPUT_DIR, name)
        if os.path.exists(path):
            print("Skipped:", name)
            continue
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Saved:", name)
    except Exception as e:
        print("Failed download:", pdf_url, e)



Visiting section page: https://manshurat.org/taxonomy/term/16
Visiting section page: https://manshurat.org/taxonomy/term/16?page=1
Visiting section page: https://manshurat.org/taxonomy/term/16?page=2
Visiting section page: https://manshurat.org/taxonomy/term/16?page=3
Visiting section page: https://manshurat.org/taxonomy/term/16?page=4
Visiting section page: https://manshurat.org/taxonomy/term/16?page=5
Visiting section page: https://manshurat.org/taxonomy/term/16?page=6
Visiting section page: https://manshurat.org/taxonomy/term/16?page=7
Visiting section page: https://manshurat.org/taxonomy/term/16?page=8
Visiting section page: https://manshurat.org/taxonomy/term/16?page=156
Visiting section page: https://manshurat.org/taxonomy/term/16?page=9
Visiting section page: https://manshurat.org/taxonomy/term/16?page=10


KeyboardInterrupt: 

In [12]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import warnings

# ====== تجاهل تحذيرات RuntimeWarning في Jupyter ======
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ====== إعدادات ======
BASE = "https://manshurat.org"
START_PAGE = "https://manshurat.org/taxonomy/term/16?page=1"
OUTPUT_DIR = "manshurat_pdfs"
HEADERS = {"User-Agent": "pdf-downloader/1.0"}
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ====== دوال مساعدة ======
def get_article_links(page_url):
    """اجمع روابط المقالات داخل صفحة القسم"""
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    # كل المقالات غالبًا داخل div.views-row أو article
    for container in soup.select("div.views-row, article"):
        a_tag = container.find("a", href=True)
        if a_tag:
            full = urljoin(BASE, a_tag['href'])
            links.append(full)
    return links

def get_pdf_links(article_url):
    """اجمع روابط PDF داخل صفحة المقالة"""
    resp = requests.get(article_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    # غالبًا الرابط موجود في <a> يحتوي على .pdf
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if ".pdf" in href.lower():   # أي رابط فيه pdf
            full = urljoin(BASE, href)
            links.append(full)
    return links

def get_pagination_links(page_url):
    """اجمع روابط الصفحات التالية (pagination)"""
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")
    pages = []
    for a in soup.select("ul.pagination li a[href]"):
        full = urljoin(BASE, a["href"])
        pages.append(full)
    return pages

# ====== زحف صفحات القسم ======
visited_pages = set()
to_visit_pages = [START_PAGE]
all_articles = set()

while to_visit_pages:
    url = to_visit_pages.pop(0)
    if url in visited_pages:
        continue
    visited_pages.add(url)
    print("Visiting section page:", url)
    try:
        # جمع المقالات
        articles = get_article_links(url)
        for a in articles:
            all_articles.add(a)
        # جمع صفحات pagination
        next_pages = get_pagination_links(url)
        for np in next_pages:
            if np not in visited_pages and np not in to_visit_pages:
                to_visit_pages.append(np)
    except Exception as e:
        print("Failed page:", url, e)
    time.sleep(0.5)  # تأخير بسيط لتقليل الضغط

print(f"Found {len(all_articles)} articles. Now fetching PDFs...")

# ====== زيارة كل مقال لجمع PDF ======
all_pdfs = set()
for article in all_articles:
    try:
        pdfs = get_pdf_links(article)
        all_pdfs.update(pdfs)
    except Exception as e:
        print("Failed article:", article, e)
    time.sleep(0.2)

print(f"Found {len(all_pdfs)} PDF files. Downloading...")

# ====== تحميل الملفات ======
for pdf_url in all_pdfs:
    try:
        r = requests.get(pdf_url, headers=HEADERS, stream=True, timeout=60)
        r.raise_for_status()
        name = os.path.basename(pdf_url)
        path = os.path.join(OUTPUT_DIR, name)
        if os.path.exists(path):
            print("Skipped:", name)
            continue
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Saved:", name)
    except Exception as e:
        print("Failed download:", pdf_url, e)


Visiting section page: https://manshurat.org/taxonomy/term/16?page=1
Visiting section page: https://manshurat.org/taxonomy/term/16
Visiting section page: https://manshurat.org/taxonomy/term/16?page=2
Visiting section page: https://manshurat.org/taxonomy/term/16?page=3
Visiting section page: https://manshurat.org/taxonomy/term/16?page=4
Visiting section page: https://manshurat.org/taxonomy/term/16?page=5
Visiting section page: https://manshurat.org/taxonomy/term/16?page=6


KeyboardInterrupt: 

In [13]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import warnings

# ====== تجاهل التحذيرات ======
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ====== إعدادات ======
BASE = "https://manshurat.org"
START_PAGE = "https://manshurat.org/taxonomy/term/16"
OUTPUT_DIR = "manshurat_pdfs"
HEADERS = {"User-Agent": "pdf-downloader/1.0"}
MAX_DEPTH = 4  # زحف من الدرجة 4
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ====== دوال مساعدة ======
def get_links(url):
    """اجمع كل الروابط داخل صفحة"""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        links = []
        for a in soup.find_all("a", href=True):
            full = urljoin(BASE, a["href"])
            links.append(full)
        return links
    except:
        return []

def is_pdf(url):
    return ".pdf" in url.lower()

def is_article(url):
    return "/node/" in url

# ====== زحف متعدد المستويات ======
visited = set()
queue = [(START_PAGE, 0)]
pdf_links = set()

while queue:
    url, depth = queue.pop(0)
    if url in visited:
        continue
    visited.add(url)
    print(f"Visiting (depth {depth}): {url}")

    try:
        links = get_links(url)
        for link in links:
            if is_pdf(link):
                pdf_links.add(link)
            elif depth + 1 < MAX_DEPTH:
                queue.append((link, depth + 1))
    except Exception as e:
        print("Failed:", url, e)
    time.sleep(0.3)

print(f"Found {len(pdf_links)} PDF files. Downloading...")

# ====== تحميل ملفات PDF ======
for pdf_url in pdf_links:
    try:
        r = requests.get(pdf_url, headers=HEADERS, stream=True, timeout=60)
        r.raise_for_status()
        name = os.path.basename(pdf_url)
        path = os.path.join(OUTPUT_DIR, name)
        if os.path.exists(path):
            print("Skipped:", name)
            continue
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Saved:", name)
    except Exception as e:
        print("Failed download:", pdf_url, e)


Visiting (depth 0): https://manshurat.org/taxonomy/term/16
Visiting (depth 1): https://manshurat.org#main-content
Visiting (depth 1): https://manshurat.org/
Visiting (depth 1): https://manshurat.org/search
Visiting (depth 1): https://manshurat.org
Visiting (depth 1): https://manshurat.org/taxonomy/term/1
Visiting (depth 1): https://manshurat.org/taxonomy/term/2
Visiting (depth 1): https://manshurat.org/taxonomy/term/4
Visiting (depth 1): https://manshurat.org/taxonomy/term/3
Visiting (depth 1): https://manshurat.org/taxonomy/term/21
Visiting (depth 1): https://manshurat.org/taxonomy/term/23
Visiting (depth 1): https://manshurat.org/taxonomy/term/24
Visiting (depth 1): https://manshurat.org/taxonomy/term/25
Visiting (depth 1): https://manshurat.org/taxonomy/term/22
Visiting (depth 1): https://manshurat.org/taxonomy/term/5
Visiting (depth 1): https://manshurat.org/taxonomy/term/7
Visiting (depth 1): https://manshurat.org/taxonomy/term/6
Visiting (depth 1): https://manshurat.org/taxonomy/

KeyboardInterrupt: 

In [14]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import warnings

# ====== تجاهل تحذيرات Jupyter ======
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ====== إعدادات ======
BASE = "https://manshurat.org"
START_PAGE = "https://manshurat.org/taxonomy/term/16"
OUTPUT_DIR = "manshurat_pdfs"
HEADERS = {"User-Agent": "pdf-downloader/1.0"}
MAX_DEPTH = 4
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ====== دوال مساعدة ======
def get_links(url):
    """اجمع كل الروابط داخل صفحة"""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        links = []
        for a in soup.find_all("a", href=True):
            full = urljoin(BASE, a["href"])
            links.append(full)
        return links
    except:
        return []

def is_pdf(url):
    return ".pdf" in url.lower()

def is_article(url):
    return "/node/" in url

def is_valid_link(url, depth):
    """تحديد الروابط الصالحة للزحف حسب العمق"""
    if is_pdf(url):
        return True
    if depth == 0 or depth == 1:
        # فقط روابط القسم 16 وpagination
        return "/taxonomy/term/16" in url
    if depth >= 2:
        # صفحات المقال
        return is_article(url)
    return False

# ====== زحف متعدد المستويات ======
visited = set()
queue = [(START_PAGE, 0)]
pdf_links = set()

while queue:
    url, depth = queue.pop(0)
    if url in visited:
        continue
    visited.add(url)
    print(f"Visiting (depth {depth}): {url}")

    try:
        links = get_links(url)
        for link in links:
            if not is_valid_link(link, depth):
                continue
            if is_pdf(link):
                pdf_links.add(link)
            elif depth + 1 < MAX_DEPTH:
                queue.append((link, depth + 1))
    except Exception as e:
        print("Failed:", url, e)
    time.sleep(0.3)

print(f"Found {len(pdf_links)} PDF files. Downloading...")

# ====== تحميل ملفات PDF ======
for pdf_url in pdf_links:
    try:
        r = requests.get(pdf_url, headers=HEADERS, stream=True, timeout=60)
        r.raise_for_status()
        name = os.path.basename(pdf_url)
        path = os.path.join(OUTPUT_DIR, name)
        if os.path.exists(path):
            print("Skipped:", name)
            continue
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Saved:", name)
    except Exception as e:
        print("Failed download:", pdf_url, e)


Visiting (depth 0): https://manshurat.org/taxonomy/term/16
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20ASC
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20DESC
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=1
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=2
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=3
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=4
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=5
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=6
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=7
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=8
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=156
Visiting (depth 2): https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20ASC&page=1
Visiting (depth 2): https://manshurat.org/taxonomy/t

KeyboardInterrupt: 

In [15]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import warnings

# ====== تجاهل تحذيرات ======
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ====== إعدادات ======
BASE = "https://manshurat.org"
START_PAGE = "https://manshurat.org/taxonomy/term/16"
OUTPUT_DIR = "manshurat_pdfs"
HEADERS = {"User-Agent": "pdf-downloader/1.0"}
MAX_DEPTH = 4
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ====== دوال مساعدة ======
def get_links(url):
    """جمع كل الروابط داخل صفحة"""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        links = []
        for a in soup.find_all("a", href=True):
            full = urljoin(BASE, a["href"])
            links.append(full)
        return links
    except:
        return []

def is_pdf(url):
    return ".pdf" in url.lower()

def is_article(url):
    return "/node/" in url

def is_valid_link(url, depth):
    """تحديد الروابط الصالحة للزحف حسب العمق"""
    if is_pdf(url):
        return True
    if depth == 0 or depth == 1:
        # فقط روابط القسم 16 و pagination
        return "/taxonomy/term/16" in url
    if depth >= 2:
        # صفحات المقال
        return is_article(url)
    return False

def get_pdfs_from_article(url):
    """جمع روابط PDF من صفحة المقال"""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        pdfs = set()
        for a in soup.find_all("a", href=True):
            href = urljoin(BASE, a["href"])
            if is_pdf(href):
                pdfs.add(href)
        return pdfs
    except:
        return set()

# ====== زحف متعدد المستويات ======
visited = set()
queue = [(START_PAGE, 0)]
pdf_links = set()

while queue:
    url, depth = queue.pop(0)
    if url in visited:
        continue
    visited.add(url)
    print(f"Visiting (depth {depth}): {url}")

    try:
        if depth == 3:
            # استخراج روابط PDF من المقالات
            pdfs_in_article = get_pdfs_from_article(url)
            pdf_links.update(pdfs_in_article)
        else:
            links = get_links(url)
            for link in links:
                if not is_valid_link(link, depth):
                    continue
                if is_pdf(link):
                    pdf_links.add(link)
                elif depth + 1 < MAX_DEPTH:
                    queue.append((link, depth + 1))
    except Exception as e:
        print("Failed:", url, e)
    time.sleep(0.3)

print(f"Found {len(pdf_links)} PDF files. Downloading...")

# ====== تحميل ملفات PDF ======
for pdf_url in pdf_links:
    try:
        r = requests.get(pdf_url, headers=HEADERS, stream=True, timeout=60)
        r.raise_for_status()
        name = os.path.basename(pdf_url)
        path = os.path.join(OUTPUT_DIR, name)
        if os.path.exists(path):
            print("Skipped:", name)
            continue
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Saved:", name)
    except Exception as e:
        print("Failed download:", pdf_url, e)


Visiting (depth 0): https://manshurat.org/taxonomy/term/16
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20ASC
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20DESC
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=1
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=2
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=3
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=4
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=5
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=6
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=7
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=8
Visiting (depth 1): https://manshurat.org/taxonomy/term/16?page=156
Visiting (depth 2): https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20ASC&page=1
Visiting (depth 2): https://manshurat.org/taxonomy/t

In [17]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, unquote
import urllib3
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# تعطيل تحذيرات SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE = "https://manshurat.org"
START_PAGE = "https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20DESC&search_api_views_fulltext=&field_document_official_year=&field_document_official_number=&field_field_sector%5B0%5D=66"
OUTPUT_DIR = "manshurat_pdfs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def create_session():
    session = requests.Session()
    retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,/;q=0.8',
        'Accept-Language': 'ar,en-US;q=0.7,en;q=0.3',
        'Accept-Encoding': 'gzip, deflate',
        'Connection': 'keep-alive',
    })
    return session

def extract_pdf_from_google_docs(url):
    try:
        parsed = urlparse(url)
        query = dict([kv.split('=') for kv in parsed.query.split('&') if '=' in kv])
        if 'url' in query:
            pdf_url = unquote(query['url'])
            if '%' in pdf_url:
                pdf_url = unquote(pdf_url)
            return pdf_url
    except:
        return None

def get_article_links(session, page_url):
    """اجمع روابط المقالات داخل صفحة القسم"""
    resp = session.get(page_url, timeout=30, verify=False)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = set()
    for a in soup.select("h2 a, div.views-field-title a, article a"):
        href = a.get("href")
        if href:
            full = urljoin(BASE, href)
            links.add(full)
    return links

def get_pdf_links(session, article_url):
    """اجمع روابط PDF داخل صفحة المقالة"""
    resp = session.get(article_url, timeout=30, verify=False)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    pdf_links = set()

    # 1. روابط PDF مباشرة
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".pdf"):
            pdf_links.add(urljoin(BASE, href))

    # 2. Google Docs iframe
    for iframe in soup.find_all('iframe', src=True):
        src = iframe['src']
        if 'docs.google.com/gview' in src:
            pdf_url = extract_pdf_from_google_docs(src)
            if pdf_url:
                pdf_links.add(pdf_url)

    # 3. أي ملفات PDF ضمن النص
    html_text = str(soup)
    for match in re.findall(r'["\']([^"\']+\.pdf)["\']', html_text, re.IGNORECASE):
        if 'docs.google.com' not in match:
            pdf_links.add(urljoin(BASE, match))

    return pdf_links

def download_pdf(session, pdf_url):
    """تحميل ملف PDF"""
    try:
        filename = os.path.basename(urlparse(pdf_url).path)
        filepath = os.path.join(OUTPUT_DIR, filename)
        if os.path.exists(filepath):
            print("⚠ تم التخطي (موجود مسبقاً):", filename)
            return
        with session.get(pdf_url, timeout=60, verify=False, stream=True) as r:
            r.raise_for_status()
            with open(filepath, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        print("✅ تم الحفظ:", filename)
    except Exception as e:
        print("❌ فشل التحميل:", pdf_url, e)

def main():
    session = create_session()
    visited_pages = set()
    to_visit_pages = [START_PAGE]
    all_articles = set()

    # ====== جمع جميع روابط المقالات مع التعامل مع pagination ======
    while to_visit_pages:
        url = to_visit_pages.pop(0)
        if url in visited_pages:
            continue
        visited_pages.add(url)
        print("🔍 زيارة صفحة القسم:", url)
        try:
            articles = get_article_links(session, url)
            all_articles.update(articles)

            # تحقق من pagination
            resp = session.get(url, timeout=30, verify=False)
            soup = BeautifulSoup(resp.text, "html.parser")
            for a in soup.select("ul.pagination li a[href]"):
                next_page = urljoin(BASE, a["href"])
                if next_page not in visited_pages:
                    to_visit_pages.append(next_page)
        except Exception as e:
            print("❌ فشل الصفحة:", url, e)
        time.sleep(1)

    print(f"📄 تم العثور على {len(all_articles)} مقال. الآن جمع ملفات PDF...")

    # ====== زيارة كل مقال لجمع PDF ======
    all_pdfs = set()
    for i, article in enumerate(all_articles, 1):
        try:
            pdfs = get_pdf_links(session, article)
            all_pdfs.update(pdfs)
        except Exception as e:
            print("❌ فشل المقال:", article, e)
        time.sleep(0.5)

    print(f"📄 تم العثور على {len(all_pdfs)} ملفات PDF. جاري التحميل...")

    # ====== تحميل الملفات ======
    for pdf_url in all_pdfs:
        download_pdf(session, pdf_url)
        time.sleep(1)

    print(f"\n✅ اكتملت العملية! جميع الملفات محفوظة في: {os.path.abspath(OUTPUT_DIR)}")

if __name__ == "__main__":
    main()


🔍 زيارة صفحة القسم: https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20DESC&search_api_views_fulltext=&field_document_official_year=&field_document_official_number=&field_field_sector%5B0%5D=66
🔍 زيارة صفحة القسم: https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20DESC&search_api_views_fulltext=&field_document_official_year=&field_document_official_number=&field_field_sector%5B0%5D=66&page=1
🔍 زيارة صفحة القسم: https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20DESC&search_api_views_fulltext=&field_document_official_year=&field_document_official_number=&field_field_sector%5B0%5D=66&page=2
🔍 زيارة صفحة القسم: https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20DESC&search_api_views_fulltext=&field_document_official_year=&field_document_official_number=&field_field_sector%5B0%5D=66&page=3
🔍 زيارة صفحة القسم: https://manshurat.org/taxonomy/term/16?sort=search_api_aggregation_1%20DESC&search_api_views_fulltext=&

KeyboardInterrupt: 

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import re

BASE = "https://manshurat.org"
START_PAGE = "https://manshurat.org/taxonomy/term/16"
OUTPUT_DIR = "manshurat_pdfs"
HEADERS = {"User-Agent": "pdf-downloader/1.0"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_article_links(page_url):
    """اجمع روابط المقالات داخل صفحة القسم"""
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    for a in soup.select("h2 a, div.views-field-title a, article a"):
        href = a.get("href")
        if href:
            full = urljoin(BASE, href)
            links.append(full)
    return links

def get_pdf_links(article_url):
    """اجمع روابط PDF داخل صفحة المقالة"""
    resp = requests.get(article_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    # تحقق من وجود iframe Google Docs
    for iframe in soup.find_all("iframe", src=True):
        src = iframe["src"]
        if "docs.google.com/gview" in src:
            match = re.search(r"url=(.+)", src)
            if match:
                pdf_url = match.group(1)
                pdf_url = requests.utils.unquote(pdf_url)
                links.append(pdf_url)
    # تحقق من روابط PDF مباشرة
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".pdf"):
            links.append(urljoin(article_url, href))
    return links

def download_pdf(pdf_url):
    """تحميل PDF"""
    try:
        filename = os.path.basename(pdf_url)
        filename = re.sub(r'[^\w\-.()]', '_', filename)
        path = os.path.join(OUTPUT_DIR, filename)
        if os.path.exists(path):
            print(f"Skipped: {filename}")
            return
        resp = requests.get(pdf_url, headers=HEADERS, stream=True, timeout=60)
        resp.raise_for_status()
        with open(path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        size = os.path.getsize(path)
        print(f"Saved: {filename} ({size:,} bytes)")
    except Exception as e:
        print(f"Failed download: {pdf_url} - {e}")

# ====== جمع صفحات القسم ======
visited_pages = set()
to_visit_pages = [START_PAGE]

while to_visit_pages:
    page_url = to_visit_pages.pop(0)
    if page_url in visited_pages:
        continue
    visited_pages.add(page_url)
    print(f"🔍 زيارة صفحة القسم: {page_url}")
    try:
        articles = get_article_links(page_url)
        for article in articles:
            print(f"📄 معالجة المقال: {article}")
            try:
                pdfs = get_pdf_links(article)
                for pdf in pdfs:
                    download_pdf(pdf)  # تحميل مباشر أثناء جمع الروابط
            except Exception as e:
                print(f"❌ فشل في المقال: {article} - {e}")

        # التحقق من وجود pagination
        resp = requests.get(page_url, headers=HEADERS, timeout=30)
        soup = BeautifulSoup(resp.text, "html.parser")
        for a in soup.select("ul.pagination li a[href]"):
            next_page = urljoin(BASE, a["href"])
            if next_page not in visited_pages and next_page not in to_visit_pages:
                to_visit_pages.append(next_page)

    except Exception as e:
        print(f"❌ فشل في صفحة القسم: {page_url} - {e}")
    time.sleep(1)

print("✅ اكتملت عملية التحميل لجميع PDF")


🔍 زيارة صفحة القسم: https://manshurat.org/taxonomy/term/16
📄 معالجة المقال: https://manshurat.org/content/dm-dstwry-tthbyt-ljr-fy-nzm-lyjr-lqdym-lqnwn-136-lsn-1981
Saved: hkm_ldstwry_lyjr_lqdym_9_nwfmbr_2024_0.pdf (3,688,201 bytes)
📄 معالجة المقال: https://manshurat.org/content/dm-qbwl-tn-fy-dstwry-fqrtyn-blqnwn-rqm-52-lsn-1969-fy-shn-yjr-lmkn-wtnzym-llq-byn-lmwjryn
Saved: 1-brqm_18_lsn_32_qdyy.pdf (146,407 bytes)
📄 معالجة المقال: https://manshurat.org/content/dm-qbwl-tn-fy-dstwry-nsws-bqrrt-lwzyr-ltlym-lly-wllyh-ltnfydhy-lqnwn-tnzym-ljmt-bshn-ltlym
Saved: 2-brqm_58_lsn_38_qdyy.pdf (224,799 bytes)
📄 معالجة المقال: https://manshurat.org/content/dm-qbwl-tn-fy-dstwry-lmd-29-mn-lqnwn-49-lsn-1977-bshn-tjyr-wby-lmkn-wtnzym-llq-byn-lmwjr
Saved: 3-brqm_59_lsn_40_qdyy.pdf (171,405 bytes)
📄 معالجة المقال: https://manshurat.org/content/dm-qbwl-tn-mn-mrtd-mnswr-fy-dstwry-nsh-grf-lsn-llm-lmryy-wlmsmw-bqnwn-tnzym-lsn-wtshjyh-bd
Saved: 4-brqm_108_lsn_40_qdyy.pdf (193,929 bytes)
📄 معالجة المقال: https

In [ ]:
import shutil

shutil.make_archive("downloads_zip", 'zip', "downloads")

In [ ]:
from google.colab import files

files.download("downloads_zip.zip")